## Data generation 

This notebook generates bakery data based on databricks samples.bakehouse dataset 

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog name")
dbutils.widgets.text("schema", "lakeflow_examples", "Schema name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")

#### Classes

Classes for generating the data.

- DataGenerator: will generate data based on a given Delta table. The generated data will have the same schema as the source table
- DatasourceConfig: configuration for the source table
- GeneratorConfig: configuration for the DataGenerator

In [0]:
from dataclasses import dataclass
import csv
import time

@dataclass
class DatasourceConfig:
    source_catalog: str
    source_schema: str
    source_table: str

    def table_path(self) -> str:
        return f"{self.source_catalog}.{self.source_schema}.{self.source_table}"

class DataGenerator:
    def __init__(self, conf: DatasourceConfig):
        self.conf = conf

    def generate(self, 
                 batch_size: int, 
                 target_dir: str, 
                 batch_interval: float = 0.1,
                 drop_columns: list[str] = None,
                 renamed_columns: dict[str, str] = None):
        if batch_size <= 0:
            raise ValueError("Batch size must be greater than 0.")

        if batch_interval <= 0:
            raise ValueError("Batch interval must be greater than 0.")

        if not target_dir:
            raise ValueError("Target directory must be specified.")

        source_table = self.conf.table_path()
        source_df = spark.read.table(source_table)
        total_rows = source_df.count()

        if drop_columns:
            source_df = source_df.drop(*drop_columns)

        if renamed_columns:
            for old_name, new_name in renamed_columns.items():
                source_df = source_df.withColumnRenamed(old_name, new_name)

        columns = source_df.columns  # Compute schema once before the loop

        n = total_rows // batch_size
        if total_rows % batch_size != 0:
            n += 1

        data = source_df.collect()
        for i in range(n):
            batch = data[i * batch_size: (i + 1) * batch_size]
            target_file = f"{target_dir.rstrip('/')}/{self.conf.source_table}_{i}.csv"
            self._write_batch(target_file, batch, columns, batch_interval)
            time.sleep(batch_interval)

    def _write_batch(self, target_file: str, data: list, columns: list, batch_interval: float):
        if not data:
            raise ValueError("Data must be specified.")
        
        if columns is None:
            raise ValueError("Columns must be specified.")

        with open(target_file, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=columns)
            writer.writeheader()
            for row in data:
                writer.writerow(row.asDict())

In [0]:
# dbutils.fs.mkdirs("/Volumes/workspace/lakeflow_examples/bakery")
dbutils.fs.ls("/Volumes/workspace/lakeflow_examples/bakery/sales")

# dbutils.fs.rm("/Volumes/workspace/lakeflow_examples/bakery", True)

### Generate sales data  

In [0]:
conf = DatasourceConfig(catalog, schema, "sales_transactions")
target_dir = "/Volumes/workspace/lakeflow_examples/bakery/sales"

dbutils.fs.mkdirs(target_dir)
DataGenerator(conf).generate(
    batch_size=100, 
    target_dir = target_dir, 
    batch_interval= 0.1,
    renamed_columns = {
            "dateTime": "updatedAt",
        }
    )

In [0]:
conf = DatasourceConfig(catalog, schema, "sales_customers")

target_dir = "/Volumes/workspace/lakeflow_examples/bakery/customers"
dbutils.fs.mkdirs(target_dir)

DataGenerator(conf).generate(
    batch_size=100, 
    target_dir= target_dir, 
    batch_interval= 0.1)

In [0]:
dbutils.fs.ls(target_dir)